# Imports

In [ ]:
# Import necessary libraries
import csv
import random

import matplotlib.pyplot as plt
import numpy as np
from proto_tools.tools.masked_models.masking import MaskingStrategy

from proto_language.language.constraint import gc_content_constraint

# Import proto-language modules
from proto_language.language.core import (
    Constraint,
    Construct,
    Program,
    Segment,
    Sequence,
)
from proto_language.language.generator import (
    RandomNucleotideGenerator,
    RandomNucleotideGeneratorConfig,
)
from proto_language.language.optimizer import (
    MCMCOptimizer,
    MCMCOptimizerConfig,
)

random.seed(42)
np.random.seed(42)

In [ ]:
def analyze_gc_content(sequence: str) -> float:
    """Calculate the GC content of a DNA sequence."""
    gc_count = 0
    for nucleotide in sequence:
        if nucleotide in "GC":
            gc_count += 1

    return (gc_count / len(sequence)) * 100 if len(sequence) > 0 else 0.0

# Write Program

In [ ]:
# Select parameters
SEQUENCE_LENGTH = 1000 # Length of DNA sequence
NUM_MCMC_STEPS = 200000  # Number of MCMC steps to run
MIN_GC = 80  # Min target for high GC content (%)
MAX_GC = 90  # Max target for high GC content (%)


# Initialize Segment
segment = Segment(length=SEQUENCE_LENGTH, sequence_type="dna")

# Initialize Construct
construct = Construct([segment])

# Initialize Generator
random_nuc_gen_config = RandomNucleotideGeneratorConfig(
    masking_strategy=MaskingStrategy(num_mutations=1),
)
random_nuc_gen = RandomNucleotideGenerator(random_nuc_gen_config)

# Assign segment to generator
random_nuc_gen.assign(segment)

# Initialize Constraint
gc_constraint = Constraint(
    inputs=[segment],
    function=gc_content_constraint,
    function_config={"min_gc": MIN_GC, "max_gc": MAX_GC},
)

# Custom logging function
def custom_logging(step: int, outputs: tuple[Segment]) -> None:
    output_sequence: Sequence = outputs[0].candidate_sequences[0]
    gc_content = output_sequence.metadata.get('gc_content', 'N/A')
    print(
        f"Custom Log - Step {step} | "
        f"sequence: {output_sequence.sequence[:50]}..., "
        f"gc_content: {gc_content}"
    )

# Initialize Optimizer
optimizer_config = MCMCOptimizerConfig(
    num_results=1,
    num_steps=NUM_MCMC_STEPS,
    max_temperature=1.0,
)

optimizer = MCMCOptimizer(
    constructs=[construct],
    generators=[random_nuc_gen],
    constraints=[gc_constraint],
    config=optimizer_config,
    custom_logging=custom_logging,
)

# Initialize Program
program = Program(
    optimizers=[optimizer],
    num_results=1,
)

# Run Program

In [ ]:
initial_construct = optimizer.history[0][0] if optimizer.history else construct
program.run()
print(f"Generated {len(optimizer.history)} sequence snapshots during optimization")
final_construct = program.constructs[0]

# Save Metadata

In [ ]:
metadata = final_construct.joined_sequences[0]._metadata
print(metadata)

# Save metadata to CSV (create directory if needed)
import os

os.makedirs('metadata_examples', exist_ok=True)
with open('metadata_examples/toy_example_metadata.csv', 'w') as f:
    writer = csv.DictWriter(f, fieldnames=metadata.keys())
    writer.writeheader()
    writer.writerow(metadata)

# Visualize

In [ ]:
# Extract history from optimizer
# Each entry in history is a dictionary with keys: "time_step", "energy_scores", "constructs"
energy_history = []
steps_history = []
gc_content_history = []

for history_entry in optimizer.history:
    # Access the constructs list from the dictionary
    construct = history_entry["constructs"][0]  # First construct
    seq = construct.joined_sequences[0]  # First sequence in batch

    # Get energy from the history entry (more direct than metadata)
    energy = history_entry["energy_scores"][0]
    energy_history.append(energy)

    # Get step number from the history entry
    step = history_entry["time_step"]
    steps_history.append(step)

    # Calculate GC content
    gc = analyze_gc_content(seq.sequence)
    gc_content_history.append(gc)

# Plot the evolution of GC content
plt.figure(figsize=(12, 8))

# GC content evolution
plt.subplot(2, 1, 1)
plt.plot(steps_history, gc_content_history, 'b-', linewidth=2)
plt.axhspan(MIN_GC, MAX_GC, color='g', alpha=0.2, label='Target GC Range')
plt.xlabel('MCMC Steps', fontsize=12)
plt.ylabel('GC Content (%)', fontsize=12)
plt.title('GC Content throughout MCMC Loop', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tick_params(labelsize=10)

# Energy evolution
plt.subplot(2, 1, 2)
plt.plot(steps_history, energy_history, 'r-', linewidth=2)
plt.xlabel('MCMC Steps', fontsize=12)
plt.ylabel('Energy (lower is better)', fontsize=12)
plt.title('Energy throughout MCMC Loop', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tick_params(labelsize=10)

# Add a main title for the entire figure
plt.suptitle('MCMC Optimization Progress', fontsize=16, y=0.98)

# Save the figure
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to make room for suptitle
plt.show()